# Delta Lake Fundamentals

Delta Lake is an open-source storage layer providing ACID transactions, schema enforcement, and time travel on Parquet files. It is the default format in Databricks. In this module, you'll learn CRUD operations, MERGE INTO, schema management, and data versioning — key exam topics.

| Exam Domain | Weight |
|---|---|
| ELT with Spark SQL and Python | 29% |
| Data Governance | 17% |

---

## Learning Objectives

After completing this module you will be able to:

- **Describe** how Delta Lake uses transaction logs to provide ACID guarantees
- **Use** `DESCRIBE HISTORY` to inspect table version history
- **Perform** time-travel queries with `VERSION AS OF` and `TIMESTAMP AS OF`
- **Execute** `OPTIMIZE` and `VACUUM` to maintain table health
- **Explain** the difference between Managed and External Delta tables

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

# Display user context
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

### Example: Creating the First Delta Table

**Objective:** Demonstration of creating a Delta table and basic properties

**Approach:**
1. Load data from Unity Catalog Volume
2. Create a managed table in Delta format
3. Explore Delta Log and metadata

In [0]:
# Load customer data from Unity Catalog Volume
customers_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATASET_PATH}/customers/customers.csv")
)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [0]:
# Create managed Delta table
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")

In [0]:
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").limit(5))

### Example: Schema Enforcement in Action

**Objective:** Demonstration of automatic schema validation during data insertion



Schema Enforcement is a critical feature that prevents "garbage in, garbage out" scenarios. Delta Lake compares incoming data schema with the target table schema and **rejects incompatible writes**.

In [0]:
# Check current table schema
spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").printSchema()

Now we will insert data. Note that in the `INSERT` query we omit `order_sk` and `order_date` columns:
- `order_sk`: will be generated automatically (unique number)
- `order_date`: will be calculated based on `order_timestamp`

In [0]:
df = spark.table("customers_delta")
display(df.orderBy("customer_id"))

In [0]:
cleaned_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta").na.drop(subset=["customer_id"])
display(cleaned_df.orderBy("customer_id"))

In [0]:
%sql

delete from customers_delta
where customer_id is null

In [0]:
# Attempt to insert invalid data (customer_id does not start with CUST)
try:
    spark.sql(f"""
        INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment)
        VALUES ('INVALID123', 'Bad', 'Customer', 'bad@example.com', '+48 000 000 000', 'Test', 'TS', 'Poland', '2023-01-01', 'Basic')
    """)
except Exception as e:
    print(f"Expected Data Quality error:\n{str(e)[:300]}...")

In [0]:
spark.sql(f"""
  INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.customers_delta (customer_id, first_name, last_name, email, phone, city, state, country, registration_date, customer_segment)
  VALUES ('CUST123', 'Bad', 'Customer', 'bad@example.com', '+48 000 000 000', 'Test', 'TS', 'Poland', '2023-01-01', 'Basic')""")

## CRUD Operations & MERGE

INSERT, UPDATE, DELETE and MERGE INTO operations on Delta tables. MERGE is the most important for the exam — it appears on almost every test.

---

**Theoretical Introduction:**

Delta Lake supports the full range of CRUD operations (Create, Read, Update, Delete), making it ideal for transactional workloads in Data Lake. All operations are:
- **Atomic**: Either fully complete or fully rolled back
- **ACID-compliant**: Ensuring data consistency
- **Recorded in Delta Log**: Full audit trail of all changes

Additionally, Delta Lake provides the powerful **MERGE INTO** operation (also known as "upsert") that combines INSERT and UPDATE in a single atomic transaction - essential for CDC (Change Data Capture) scenarios.

### Example: INSERT Operation

**Objective:** Adding new records to an existing table

In [0]:
# Verify insertion
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_id").like("CUST02%"))
    .orderBy("customer_id")
)

### Example: UPDATE Operation

**Objective:** Updating existing records in the table

### Example: DELETE Operation

**Objective:** Deleting records from a Delta table

In [0]:
# DELETE specific customer
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers_delta
    WHERE customer_id = 'CUST020002'
""")

In [0]:
# Verify deletion
deleted_check = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta") \
    .filter(F.col("customer_id") == "CUST020002") \
    .count()

display(
    spark.createDataFrame([
        ("Records with customer_id CUST020002", deleted_check)
    ], ["description", "count"])
)

### Example: MERGE INTO (Upsert)

**Objective:** Demonstration of upsert operation - update existing and insert new records in a single atomic transaction

**MERGE INTO syntax:**

```sql
MERGE INTO target
USING source ON target.key = source.key
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT (...) VALUES (...)
WHEN NOT MATCHED BY SOURCE THEN DELETE -- optional
```

| Clause | Description |
|---|---|
| `WHEN MATCHED` | Executed when a source row matches a target row (UPDATE or DELETE) |
| `WHEN NOT MATCHED` | Executed when the source row has no match in target (INSERT) |
| `WHEN NOT MATCHED BY SOURCE` | Executed when the target row has no match in source (DELETE) |
| `AND condition` | Optional extra filter on any `WHEN` clause |

MERGE INTO is especially useful when processing changes from transactional systems (CDC patterns). It allows you to:
- **Update** existing records when a match is found
- **Insert** new records when no match exists
- **Delete** records based on conditions (optional)


<img src="../../assets/images/ff01677d3d4a45d6a6a7530d8911b785.png" width="800">

In [0]:
# Verify MERGE results
display(
    spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers_delta")
    .filter(F.col("customer_id").isin(["CUST010001", "CUST030001", "CUST030002"]))
    .orderBy("customer_id")
)

## Metadata and Analytics

DESCRIBE DETAIL, DESCRIBE HISTORY and Delta Log internals. Understanding metadata commands is essential for auditing, debugging, and compliance on the exam.

---

| Command | Signature | Description |
|---|---|---|
| `DESCRIBE DETAIL` | `DESCRIBE DETAIL table` | Returns file count, size, location, partitioning, table properties |
| `DESCRIBE HISTORY` | `DESCRIBE HISTORY table [LIMIT n]` | Full audit log: operation, user, timestamp, metrics per version |
| `SHOW TBLPROPERTIES` | `SHOW TBLPROPERTIES table` | Lists all key-value table properties (e.g., CDF enabled, constraints) |
| `DESCRIBE EXTENDED` | `DESCRIBE EXTENDED table` | Schema + storage info + table properties in one output |

**Theoretical Introduction:**

Delta Lake offers rich metadata about tables and operations that enables:
- **Auditing**: Who changed what and when
- **Debugging**: Understanding operation performance and metrics
- **Compliance**: Meeting regulatory requirements for data lineage

### Example: DESCRIBE DETAIL

**Objective:** Analysis of Delta table metadata and physical storage details

In [0]:
# Detailed table information
display(
    spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.customers_delta")
)

### Example: Operation History Analysis

**Objective:** Deeper analysis of history and operation metrics

In [0]:
# History with additional metrics
history_df = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers_delta")

display(
    history_df.select(
        "version", 
        "timestamp", 
        "operation", 
        "operationMetrics.numTargetRowsInserted",
        "operationMetrics.numTargetRowsUpdated",
        "operationMetrics.numTargetRowsDeleted"
    )
)

In [0]:
# Get table path
table_details = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.customers_delta")
display(table_details.select("location", "numFiles", "sizeInBytes"))

In [0]:
# View complete Delta table history
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers_delta"))

### Demo: Time Travel Setup

In [0]:
spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo """)

In [0]:
# Create a new table specifically for Time Travel demonstration
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo (
    id INT,
    name STRING,
    status STRING,
    updated_at TIMESTAMP
) USING DELTA
""")

# Version 0: Insert initial data
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (1, 'Alice', 'active', current_timestamp()),
    (2, 'Bob', 'active', current_timestamp()),
    (3, 'Charlie', 'active', current_timestamp())
""")

print("Version 0: Initial data inserted")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [0]:
# Version 1: Update some records
spark.sql(f"""
UPDATE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
SET status = 'premium', updated_at = current_timestamp()
WHERE name = 'Alice'
""")

print("Version 1: Alice upgraded to premium")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [0]:
# Version 2: Insert new record
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (4, 'Diana', 'new', current_timestamp())
""")

print("Version 2: Diana added")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [0]:
# Version 3: Delete a record
spark.sql(f"""
DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo
WHERE name = 'Charlie'
""")

print("Version 3: Charlie deleted")
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

### Example: Table History Exploration

**Objective:** Use DESCRIBE HISTORY to analyze all operations on the table

In [0]:
# Show complete history of all operations
display(
    spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
)

### Example: Time Travel Queries

**Objective:** Access previous versions of data using VERSION AS OF and TIMESTAMP AS OF

In [0]:
# Access data from version 0 (initial state)
print("Version 0 - Initial data (before any changes):")
display(
    spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF 0")
)

In [0]:
# Access data from version 1 (after Alice upgrade)
print("Version 1 - After Alice upgrade:")
display(
    spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF 1")
)

In [0]:
# Compare record counts between versions
version_counts = []
for v in range(4):
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF {v}").first()[0]
    version_counts.append((f"Version {v}", count))

display(spark.createDataFrame(version_counts, ["version", "record_count"]))

### Example: Disaster Recovery - Accidental Deletion

**Objective:** Simulate accidental data deletion and recover using RESTORE

In [0]:
# DISASTER! Accidental deletion of ALL data
spark.sql(f"DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")

print("Oh no! All data deleted!")
print("Record count after deletion:", spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").count())

In [0]:
# Check history to find the last good version
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
display(history.select("version", "timestamp", "operation"))

In [0]:
# RESTORE to version before the accidental deletion
# The last good version is the one before DELETE (version 3 in our case)
last_good_version = spark.sql(f"""
    SELECT version FROM (
        SELECT version, operation, 
               ROW_NUMBER() OVER (ORDER BY version DESC) as rn
        FROM (DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo)
        WHERE operation != 'DELETE'
    ) WHERE rn = 1
""").first()[0]

In [0]:
print(f"Restoring to version: {last_good_version}")
spark.sql(f"RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo TO VERSION AS OF {last_good_version}")

In [0]:
# Verify restoration
print("Data restored successfully!")
print("Record count after RESTORE:", spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").count())
display(spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.time_travel_demo"))

In [0]:
%sql

DESCRIBE HISTORY time_travel_demo

### Example: VACUUM and Its Impact on Time Travel

**Objective:** Understand how VACUUM affects Time Travel capabilities

**Critical Concept:** VACUUM removes old data files that are no longer referenced by the current version of the table. Once vacuumed, **Time Travel to those versions becomes impossible**.

**Default Retention:** 7 days (168 hours)
- This means you can Time Travel to any version within the last 7 days
- After VACUUM, only versions within the retention period are accessible

In [0]:
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VALUES
    (4, 'Diana', 'new', current_timestamp()),
    (5, 'Eve', 'active', current_timestamp()),
    (6, 'Frank', 'active', current_timestamp())
""")

In [0]:
# Let's see what versions are available BEFORE vacuum
print("Available versions before VACUUM:")
display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo").select("version", "timestamp", "operation"))

In [0]:
# Check table size AFTER VACUUM
after_vacuum = spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo")
display(after_vacuum.select("numFiles", "sizeInBytes"))

In [0]:
# Now try to access an old version - this will fail after VACUUM!

spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.time_travel_demo VERSION AS OF 0").display()

### Example: Enabling Change Data Feed

**Objective:** Enable CDF on a Delta table to track all row-level changes

### Example: Generating and Tracking Changes

**Objective:** Perform various DML operations and observe how CDF tracks each one

### Example: Reading Change Data Feed

**Objective:** Read and analyze change data with CDF metadata columns

## Resource Cleanup

Clean up resources created during the notebook:

---
← [ELT Ingestion & Transformations](../modules/M02_elt_ingestion.ipynb) | [Data Quality](../modules/M04_data_quality.ipynb) →